In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
# skip pipelines due to version compatibility issues
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True).eval()

prompt = "Solve: If you have 3 apples and buy 2 more, how many apples do you have? Answer:"
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

W0328 17:00:45.810000 21612 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Solve: If you have 3 apples and buy 2 more, how many apples do you have? Answer: You have 5 apples. To determine the number of apples after buying 2 more, we can follow these steps:

1. Start with the initial number of apples, which is 3.
2. Add the number of apples bought, which is 2.

So, we perform the addition:
\[ 3 + 2 = 5 \]

Therefore, after buying 2 more apples, you have \(\boxed{5}\) apples.


In [ ]:
# multi turn chat - from chatGPT
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# store conversation
chat_history = [
    {"role": "system", "content": "You are a helpful assistant. Always reply in English only."}
]

def build_prompt(chat_history):
    messages = []
    for turn in chat_history:
        messages.append({
            "role": turn["role"],
            "content": turn["content"]
        })
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

print("Hello! How are you? (type 'exit' or 'quit' to end the conversation)")
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    chat_history.append({"role": "user", "content": user_input})
    
    prompt = build_prompt(chat_history)
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # extract only new response
    assistant_reply = response[len(prompt):].strip()
    
    print("Bot:", assistant_reply)
    
    chat_history.append({"role": "assistant", "content": assistant_reply})


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Hello! How are you? (type 'exit' or 'quit' to end the conversation)
Bot: 
Bot: 
Bot: 
Bot: 
Bot: 
Bot: 


In [ ]:
# edited by me
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# store conversation
chat_history = []

def build_prompt(chat_history):
    prompt = ""
    for turn in chat_history:
        if turn["role"] == "user":
            prompt += f"User: {turn['content']}\n"
        else:
            prompt += f"Assistant: {turn['content']}\n"
    prompt += "Assistant: "
    return prompt

print("Hello! Welcome to chatLLM")
while True:
    print("Please provide an input ")
    user_input = input("You: ")
    print(user_input)
    print(f"User Input: {user_input}")
    if user_input.lower() in ["exit"]:
        break
    
    chat_history.append({"role": "user", "content": user_input})
    
    prompt = build_prompt(chat_history)
    
    inputs = tokenizer(prompt, return_tensors="pt")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # extract only new response
    assistant_reply = response[len(prompt):].strip()
    
    print("Bot:", assistant_reply)
    
    chat_history.append({"role": "assistant", "content": assistant_reply})


W0328 17:01:37.029000 5824 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Hello! Welcome to chatLLM
Please provide an input 
User Input: 
Bot: "The new employee, who had been struggling to find a job after graduation, was very nervous on his first day. 
He had spent countless hours on the internet trying to find a job that would match his skills. 
But nothing seemed to be working out. 
He had tried every possible combination of job openings, but still had no luck."

I made some small changes to improve clarity and grammar, including:

- Changing "who" to "who"
Please provide an input 
User Input: hello
Bot: Hello, how can I help you today?

I rephrased the sentence to make it more friendly and less abrupt. I also changed "hello" to "how can I help you today?" to make it more inviting and clear what the user might be asking for.
Please provide an input 


In [ ]:
# GSM-8K evaluator benchmark - chatgpt
# load GSM8K dataset
from datasets import load_dataset

dataset = load_dataset("gsm8k", "main")
test_data = dataset["test"]

print(test_data[0])

In [ ]:
# load model
from transformers import pipeline
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
)

# chain of thought prompting
def format_prompt(question):
    return f"""Solve the following math problem step by step:

Question: {question}
Answer:"""

In [ ]:
# run inference on 1 example
example = test_data[0]

prompt = format_prompt(example["question"])

output = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print(output[0]["generated_text"])

In [ ]:
# extract final answer
import re

def extract_answer(text):
    match = re.search(r"#### (\d+)", text)
    return match.group(1) if match else None
print(extract_answer(output[0]["generated_text"]))

In [ ]:
from deepeval.benchmarks import GSM8K

class HFModel:
    def __init__(self, generator):
        self.generator = generator

    def generate(self, prompt: str) -> str:
        output = self.generator(
            prompt,
            max_new_tokens=150,
            do_sample=False
        )
        return output[0]["generated_text"]
hf_model = HFModel(generator)
print(hf_model.generate("What is 2+2?"))
benchmark = GSM8K(
    n_problems=10,
    n_shots=3,
    enable_cot=True
)

benchmark.evaluate(model=hf_model)
print(benchmark.overall_score)

In [ ]:
# zero shot prompting
from transformers import pipeline

# Use an instruction-tuned model
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct"
)

def zero_shot_prompt(question):
    prompt = f"""Answer the following question clearly and concisely.

Question: {question}
Answer:"""

    output = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )
  # Remove prompt from output
    full_text = output[0]["generated_text"]
    answer = full_text[len(prompt):].strip()

    return answer

zero_shot_prompt("Who is the current president of the United States?")

In [ ]:
# few shot prompting
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",   # replace later if needed
    device=-1
)

def few_shot_prompt(question):
    prompt = f"""Solve the following math problems.

Q: If you have 3 apples and buy 2 more, how many apples do you have?
A: You start with 3 apples and buy 2 more. 3 + 2 = 5. Answer: 5

Q: A shop has 10 candies. It sells 4. How many are left?
A: The shop had 10 candies and sold 4. 10 - 4 = 6. Answer: 6

Q: {question}
A:"""

    output = generator(
        prompt,
        max_new_tokens=120,
        do_sample=False
    )

    full_text = output[0]["generated_text"]

    # remove prompt
    answer = full_text[len(prompt):].strip()
    return answer
print(few_shot_prompt("If you have 5 oranges and eat 2, how many do you have left?"))